In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage.restoration import unwrap_phase

In [2]:
# Параметры восстановления полос при закрашивании
DC_EXCLUSION_RADIUS = 40
LOBE_RADIUS = 60

INPAINT_RADIUS_COMPLEX = 3
INPAINT_RADIUS_BG = 5

# Параметры БПФ для расчета фазы
CENTER_EXCLUSION_RADIUS = 60
PHASE_LOBE_RADIUS = 70
FIT_TILT_AND_OFFSET = False

PLOT_DPI = 160

In [3]:
INPUT_PATH = "/content/Img1.bmp"
OUTPUT_DIR = Path("/content/result_stripes_Img1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Параметры поиска темных пятен
BG_SIGMA = 9.0
DARK_SIGMA = 1.2
THR_K = 1.6
MIN_AREA = 15
MAX_AREA = 250
MASK_DILATE = 1
MIN_AREA_AFTER_DILATE = 50
DETREND_MODE = "spherical"
OVEREXPOSED = False

In [4]:
def fft2c(img: np.ndarray) -> np.ndarray:
    return np.fft.fftshift(np.fft.fft2(img))

def ifft2c(spec: np.ndarray) -> np.ndarray:
    return np.fft.ifft2(np.fft.ifftshift(spec))

def circular_mask(shape, center, radius):
    h, w = shape
    yy, xx = np.ogrid[:h, :w]
    cy, cx = center
    return (yy - cy) ** 2 + (xx - cx) ** 2 <= radius ** 2

def norm_to_u8(a):
    a = a.astype(np.float32)
    mn, mx = float(np.nanmin(a)), float(np.nanmax(a))
    if mx - mn < 1e-9:
        return np.zeros_like(a, dtype=np.uint8)
    a = (a - mn) / (mx - mn)
    return np.clip(a * 255, 0, 255).astype(np.uint8)

def keep_components_by_area(mask, min_area, max_area):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )
    out = np.zeros_like(mask, dtype=np.uint8)
    for i in range(1, num_labels):
        area = stats[i, cv2.CC_STAT_AREA]
        if min_area <= area <= max_area:
            out[labels == i] = 1
    return out

def inpaint_float32(img_f32, mask_u8, radius=3):
    return cv2.inpaint(img_f32.astype(np.float32), mask_u8, radius, cv2.INPAINT_TELEA)

In [5]:
# функции для работы с темными пятнами

def local_color_adapt_recon(img, recon, mask, ring_radius=18, blend_sigma=2.5):
    """
    Локально подгоняем восстановленные полосы к яркости исходного изображения по кольцу вокруг пятна
    """
    img = img.astype(np.float32)
    recon = recon.astype(np.float32)
    result = img.copy()

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )

    for i in range(1, num_labels):
        component = (labels == i).astype(np.uint8)
        area = stats[i, cv2.CC_STAT_AREA]
        if area < 5:
            continue

        x = stats[i, cv2.CC_STAT_LEFT]
        y = stats[i, cv2.CC_STAT_TOP]
        w = stats[i, cv2.CC_STAT_WIDTH]
        h = stats[i, cv2.CC_STAT_HEIGHT]

        pad = ring_radius + 5
        x1 = max(0, x - pad)
        y1 = max(0, y - pad)
        x2 = min(img.shape[1], x + w + pad)
        y2 = min(img.shape[0], y + h + pad)

        img_roi = img[y1:y2, x1:x2]
        recon_roi = recon[y1:y2, x1:x2]
        comp_roi = component[y1:y2, x1:x2].astype(bool)

        kernel = np.ones((2 * ring_radius + 1, 2 * ring_radius + 1), np.uint8)
        dilated = cv2.dilate(comp_roi.astype(np.uint8), kernel, iterations=1).astype(bool)
        ring = dilated & (~comp_roi)

        if np.count_nonzero(ring) < 20:
            continue

        # локальная подгонка recon к img по кольцу вокруг пятна:
        # img ~ gain * recon + bias
        x_fit = recon_roi[ring].ravel()
        y_fit = img_roi[ring].ravel()

        A = np.column_stack([x_fit, np.ones_like(x_fit)])
        gain, bias = np.linalg.lstsq(A, y_fit, rcond=None)[0]

        recon_local = gain * recon_roi + bias
        recon_local = np.clip(recon_local, 0, 255)

        soft = cv2.GaussianBlur(comp_roi.astype(np.float32), (0, 0), blend_sigma)
        soft = soft / (soft.max() + 1e-8)
        soft = np.clip(soft, 0.0, 1.0)

        roi_result = result[y1:y2, x1:x2]
        roi_result[comp_roi] = (
            roi_result[comp_roi] * (1.0 - soft[comp_roi])
            + recon_local[comp_roi] * soft[comp_roi]
        )

        # более плавное сглаживание
        # ALPHA = 0.35
        # blend = ALPHA * soft[comp_roi]

        # roi_result[comp_roi] = (
        #     roi_result[comp_roi] * (1.0 - blend)
        #     + recon_local[comp_roi] * blend
        # )


        result[y1:y2, x1:x2] = roi_result

    return np.clip(result, 0, 255).astype(np.uint8)

def detect_dark_spots_mask(img_u8: np.ndarray) -> np.ndarray:
    """Маска маленьких резких и больших размытых темных пятен"""
    img = img_u8.astype(np.float32)

    # маска для маленьких резких темных пятен
    background_local = cv2.GaussianBlur(img, (0, 0), BG_SIGMA)
    dark_map = background_local - img
    dark_map = cv2.GaussianBlur(dark_map, (0, 0), DARK_SIGMA)

    mu = float(dark_map.mean())
    sigma = float(dark_map.std())
    thr = mu + THR_K * sigma

    mask = (dark_map > thr).astype(np.uint8)
    mask = keep_components_by_area(mask, MIN_AREA, MAX_AREA)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((3, 3), np.uint8))
    mask = keep_components_by_area(mask, MIN_AREA, MAX_AREA)

    # маска для больших размытых пятен
    background_large = cv2.GaussianBlur(img, (0, 0), 90.0)
    img_flat = img / (background_large + 1e-6)
    img_flat = cv2.GaussianBlur(img_flat, (0, 0), 3.0)

    thr_large = np.quantile(img_flat, 0.08)
    mask_large = (img_flat < thr_large).astype(np.uint8)
    mask_large = cv2.morphologyEx(mask_large, cv2.MORPH_OPEN, np.ones((11, 11), np.uint8))
    mask_large = cv2.morphologyEx(mask_large, cv2.MORPH_CLOSE, np.ones((35, 35), np.uint8))
    mask_large = keep_components_by_area(mask_large, min_area=500, max_area=30000)

    mask = np.logical_or(mask, mask_large).astype(np.uint8)

    if MASK_DILATE > 0:
        mask = cv2.dilate(mask, np.ones((3, 3), np.uint8), iterations=MASK_DILATE)

    mask = keep_components_by_area(mask, MIN_AREA_AFTER_DILATE, MAX_AREA)
    return mask.astype(np.uint8)

def fill_dark_spots_by_stripe_reconstruction(img_u8: np.ndarray, mask: np.ndarray):
    """
    Восстанавливает полосы в маске через боковое спектральное плечо
    и вставляет восстановленный фрагмент в исходную интерферограмму
    """
    img = img_u8.astype(np.float32)
    h, w = img.shape
    cy, cx = h // 2, w // 2
    mask_u8 = (mask * 255).astype(np.uint8)
    mask_bool = mask.astype(bool)

    # БПФ, поиск бокового спектрального плеча
    F = fft2c(img)
    mag = np.log1p(np.abs(F))

    mag_search = mag.copy()
    mag_search[circular_mask(img.shape, (cy, cx), DC_EXCLUSION_RADIUS)] = 0
    py, px = np.unravel_index(np.argmax(mag_search), mag_search.shape)

    lobe_mask = circular_mask(img.shape, (py, px), LOBE_RADIUS)

    F_lobe = np.zeros_like(F, dtype=np.complex128)
    F_lobe[lobe_mask] = F[lobe_mask]

    # перенос плеча в центр
    shift_y = cy - py
    shift_x = cx - px
    F_centered = np.roll(np.roll(F_lobe, shift_y, axis=0), shift_x, axis=1)

    # комплексная огибающая
    env = ifft2c(F_centered)
    env_re = np.real(env).astype(np.float32)
    env_im = np.imag(env).astype(np.float32)

    # продолжение полос внутри пятен
    env_re_inp = inpaint_float32(env_re, mask_u8, radius=INPAINT_RADIUS_COMPLEX)
    env_im_inp = inpaint_float32(env_im, mask_u8, radius=INPAINT_RADIUS_COMPLEX)
    env_inp = env_re_inp + 1j * env_im_inp

    yy, xx = np.mgrid[:h, :w]
    ky = 2.0 * np.pi * (py - cy) / h
    kx = 2.0 * np.pi * (px - cx) / w
    carrier = np.exp(1j * (kx * xx + ky * yy))

    # полосовая часть интерферограммы
    band = 2.0 * np.real(env_inp * carrier).astype(np.float32)

    # фон
    img_inpaint_base = inpaint_float32(img, mask_u8, radius=INPAINT_RADIUS_BG)
    bg = cv2.GaussianBlur(img_inpaint_base, (0, 0), 20.0)

    # подгоняем яркость: img ~ a * band + b * bg + c
    good = ~mask_bool
    A_fit = np.column_stack([
        band[good].ravel(),
        bg[good].ravel(),
        np.ones(np.count_nonzero(good), dtype=np.float32),
    ])

    coef, _, _, _ = np.linalg.lstsq(A_fit, img[good].ravel(), rcond=None)
    a, b, c = coef

    recon = a * band + b * bg + c
    recon = np.clip(recon, 0, 255).astype(np.float32)
    recon_soft = cv2.GaussianBlur(recon.astype(np.float32), (0, 0), 0.8)

    result = local_color_adapt_recon(
        img=img,
        recon=recon_soft,
        mask=mask,
        ring_radius=18,
        blend_sigma=2.5,
    )

    return result, mag

In [6]:
# функции для двойного преобразования Фурье

def log_spectrum(F: np.ndarray) -> np.ndarray:
    return np.log1p(np.abs(F))

def find_first_order_peak(img: np.ndarray, center_exclusion_radius=CENTER_EXCLUSION_RADIUS):
    F = fft2c(img.astype(np.float32))
    mag = log_spectrum(F)
    h, w = img.shape
    cy, cx = h // 2, w // 2

    mag_search = mag.copy()
    mag_search[circular_mask(img.shape, (cy, cx), center_exclusion_radius)] = 0

    peak = np.unravel_index(np.argmax(mag_search), mag_search.shape)
    py, px = peak
    return F, mag, (py, px)

def unwrap_phase_2d(phase_wrapped: np.ndarray) -> np.ndarray:
    return unwrap_phase(phase_wrapped)

def safe_phase_difference_from_fields(field_before, field_after, mask=None):
    """
    Считаем разность фаз устойчиво
    """
    diff_wrapped = np.angle(field_after * np.conj(field_before))

    if mask is not None:
        diff_ma = np.ma.array(diff_wrapped, mask=~mask)
        diff_unwrapped = unwrap_phase(diff_ma).filled(np.nan)
    else:
        diff_unwrapped = unwrap_phase(diff_wrapped)

    return diff_wrapped, diff_unwrapped

def build_background_mask(amplitude: np.ndarray, blur_sigma: float = 6.0, q: float = 0.55) -> np.ndarray:
    """Маска фона по сглаженной амплитуде"""
    amp = amplitude.astype(np.float32)
    amp_smooth = ndi.gaussian_filter(amp, blur_sigma)

    thr = np.quantile(amp_smooth, q)
    bg_mask = amp_smooth <= thr

    bg_mask = ndi.binary_opening(bg_mask, structure=np.ones((5, 5)))
    bg_mask = ndi.binary_closing(bg_mask, structure=np.ones((7, 7)))
    bg_mask = ndi.binary_fill_holes(bg_mask)

    return bg_mask

def fit_linear_trend(phase, mask=None):
    """
    Аппроксимация фона плоскостью:
        z = ax + by + c
    """
    h, w = phase.shape
    yy, xx = np.mgrid[:h, :w]

    valid = np.isfinite(phase)
    if mask is not None:
        valid &= mask

    A = np.column_stack([
        xx[valid].ravel(),
        yy[valid].ravel(),
        np.ones(np.count_nonzero(valid))
    ])
    b = phase[valid].ravel()

    coeff, *_ = np.linalg.lstsq(A, b, rcond=None)
    plane = coeff[0] * xx + coeff[1] * yy + coeff[2]
    return plane, coeff

def fit_quadratic_background_robust(
    phase_unwrapped: np.ndarray,
    mask: np.ndarray = None,
    max_iter: int = 3,
    sigma_clip: float = 2.5,
    fit_tilt: bool = True,
):
    """
    Аппроксимация гладкого сферического фона квадратичной моделью:
        phi(x, y) = a*X^2 + b*Y^2 + c*X*Y + d*X + e*Y + f
    """
    h, w = phase_unwrapped.shape
    yy, xx = np.mgrid[:h, :w].astype(np.float64)

    x0 = (w - 1) / 2.0
    y0 = (h - 1) / 2.0
    X = xx - x0
    Y = yy - y0

    valid = np.isfinite(phase_unwrapped)
    if mask is not None:
        valid &= mask

    if valid.sum() < 50:
        raise ValueError("Слишком мало точек для оценки сферического тренда")

    Z = phase_unwrapped.copy()
    current_mask = valid.copy()

    for _ in range(max_iter):
        if fit_tilt:
            A = np.column_stack([
                X[current_mask] ** 2,
                Y[current_mask] ** 2,
                X[current_mask] * Y[current_mask],
                X[current_mask],
                Y[current_mask],
                np.ones(current_mask.sum(), dtype=np.float64),
            ])
        else:
            A = np.column_stack([
                X[current_mask] ** 2,
                Y[current_mask] ** 2,
                X[current_mask] * Y[current_mask],
                np.ones(current_mask.sum(), dtype=np.float64),
            ])

        z = Z[current_mask]
        coeff, *_ = np.linalg.lstsq(A, z, rcond=None)

        if fit_tilt:
            a, b2, c, d, e, f0 = coeff
            trend = a * X**2 + b2 * Y**2 + c * X * Y + d * X + e * Y + f0
            params = {"a": a, "b": b2, "c": c, "d": d, "e": e, "f": f0, "x0": x0, "y0": y0}
        else:
            a, b2, c, f0 = coeff
            trend = a * X**2 + b2 * Y**2 + c * X * Y + f0
            params = {"a": a, "b": b2, "c": c, "d": 0.0, "e": 0.0, "f": f0, "x0": x0, "y0": y0}

        resid = Z - trend
        med = np.median(resid[current_mask])
        mad = np.median(np.abs(resid[current_mask] - med)) + 1e-12
        robust_sigma = 1.4826 * mad

        new_mask = valid & (np.abs(resid - med) < sigma_clip * robust_sigma)
        if mask is not None:
            new_mask &= mask

        if new_mask.sum() < 100:
            break

        current_mask = new_mask

    return trend, params, current_mask

def spherical_phase_multiplier(shape, params):
    """Строит фазовый множитель exp(-i * phi_trend)"""
    h, w = shape
    yy, xx = np.mgrid[:h, :w].astype(np.float64)

    x0 = params["x0"]
    y0 = params["y0"]
    X = xx - x0
    Y = yy - y0

    phi = (
        params["a"] * X**2
        + params["b"] * Y**2
        + params["c"] * X * Y
        + params["d"] * X
        + params["e"] * Y
        + params["f"]
    )

    return np.exp(-1j * phi), phi

def remove_spherical_trend_robust(complex_field: np.ndarray, amplitude: np.ndarray = None,
                                  background_mask: np.ndarray = None, fit_tilt: bool = True):
    wrapped_phase = np.angle(complex_field)
    unwrapped_phase = unwrap_phase_2d(wrapped_phase)

    if amplitude is None:
        amplitude = np.abs(complex_field)

    if background_mask is None:
        background_mask = build_background_mask(amplitude)

    trend, params, fit_mask = fit_quadratic_background_robust(
        unwrapped_phase,
        mask=background_mask,
        max_iter=3,
        sigma_clip=2.5,
        fit_tilt=fit_tilt,
    )

    phase_mult, trend_map = spherical_phase_multiplier(complex_field.shape, params)
    corrected_field = complex_field * phase_mult

    corrected_wrapped = np.angle(corrected_field)
    corrected_unwrapped = unwrap_phase_2d(corrected_wrapped)

    return {
        "corrected_field": corrected_field,
        "original_wrapped_phase": wrapped_phase,
        "original_unwrapped_phase": unwrapped_phase,
        "background_mask": background_mask,
        "fit_mask": fit_mask,
        "spherical_trend": trend_map,
        "corrected_wrapped_phase": corrected_wrapped,
        "corrected_unwrapped_phase": corrected_unwrapped,
        "params": params,
    }

In [7]:
def reconstruct_complex_field_double_fft(
    img: np.ndarray,
    lobe_radius=PHASE_LOBE_RADIUS,
    center_exclusion_radius=CENTER_EXCLUSION_RADIUS,
    include_dc=False,
    margin_pct=0.0,
    use_spherical_detrend=False,
    fit_tilt=FIT_TILT_AND_OFFSET,
):
    """
    Двойное БПФ:
    - БПФ интерферограммы
    - выделение бокового спектрального плеча
    - перенос плеча в центр
    - обратное БПФ для комплексного поля
    - unwrap фазы
    - сферический детренд
    """
    F, mag, (py, px) = find_first_order_peak(img, center_exclusion_radius)

    h, w = img.shape
    cy, cx = h // 2, w // 2

    r = int(lobe_radius * (1 + margin_pct))
    mask = circular_mask(img.shape, (py, px), r)

    if include_dc:
        mask |= circular_mask(img.shape, (cy, cx), center_exclusion_radius)
    else:
        mask &= ~circular_mask(img.shape, (cy, cx), center_exclusion_radius)

    mask = mask.astype(np.float32)
    sideband = F * mask

    shift_y = cy - py
    shift_x = cx - px
    sideband_shifted = np.roll(sideband, shift=(shift_y, shift_x), axis=(0, 1))

    complex_field = ifft2c(sideband_shifted)
    amplitude = np.abs(complex_field)
    wrapped_phase = np.angle(complex_field)
    unwrapped_phase = unwrap_phase_2d(wrapped_phase)

    corrected_field = complex_field
    corrected_wrapped_phase = wrapped_phase
    corrected_unwrapped_phase = unwrapped_phase
    spherical_trend = np.zeros_like(unwrapped_phase)
    background_mask = np.zeros_like(unwrapped_phase, dtype=bool)
    fit_mask = np.zeros_like(unwrapped_phase, dtype=bool)
    spherical_params = None

    if use_spherical_detrend:
        background_mask = build_background_mask(amplitude)

        detr = remove_spherical_trend_robust(
            complex_field,
            amplitude=amplitude,
            background_mask=background_mask,
            fit_tilt=fit_tilt,
        )

        corrected_field = detr["corrected_field"]
        corrected_wrapped_phase = detr["corrected_wrapped_phase"]
        corrected_unwrapped_phase = detr["corrected_unwrapped_phase"]
        spherical_trend = detr["spherical_trend"]
        background_mask = detr["background_mask"]
        fit_mask = detr["fit_mask"]
        spherical_params = detr["params"]

    return {
        "F": F,
        "log_spectrum": mag,
        "peak": (py, px),
        "mask": mask,
        "sideband": sideband,
        "sideband_shifted": sideband_shifted,
        "complex_field": complex_field,
        "amplitude": amplitude,
        "wrapped_phase": wrapped_phase,
        "unwrapped_phase": unwrapped_phase,
        "corrected_field": corrected_field,
        "corrected_wrapped_phase": corrected_wrapped_phase,
        "corrected_unwrapped_phase": corrected_unwrapped_phase,
        "spherical_trend": spherical_trend,
        "background_mask": background_mask,
        "fit_mask": fit_mask,
        "spherical_params": spherical_params,
    }

In [8]:
# восстановление изображения с помощью преобразования Фурье

def gaussian_mask(shape, center, radius):
    h, w = shape
    yy, xx = np.mgrid[:h, :w]
    cy, cx = center
    r2 = (yy - cy) ** 2 + (xx - cx) ** 2
    return np.exp(-r2 / (2 * radius ** 2))


def find_fringe_lobe(F, dc_radius=40):
    mag = np.log1p(np.abs(F))
    h, w = mag.shape
    cy, cx = h // 2, w // 2

    search = mag.copy()
    search[circular_mask(mag.shape, (cy, cx), dc_radius)] = 0

    py, px = np.unravel_index(np.argmax(search), search.shape)
    return py, px


def reconstruct_fringes_by_fourier_mask(img_u8, dc_radius=40, lobe_radius=55, keep_dc=True):
    """
    Восстановление интерференционных полос через маску в плоскости Фурье
    """
    img = img_u8.astype(np.float32)

    F = fft2c(img)
    h, w = img.shape
    cy, cx = h // 2, w // 2

    # находим боковое плечо полос
    py, px = find_fringe_lobe(F, dc_radius=dc_radius)

    # симметричное плечо
    py2 = 2 * cy - py
    px2 = 2 * cx - px

    # маски на два боковых плеча
    mask1 = gaussian_mask(img.shape, (py, px), lobe_radius)
    mask2 = gaussian_mask(img.shape, (py2, px2), lobe_radius)

    mask = mask1 + mask2

    # при необходимости оставляем низкочастотный фон
    if keep_dc:
        mask_dc = gaussian_mask(img.shape, (cy, cx), dc_radius)
        mask = mask + mask_dc

    mask = np.clip(mask, 0, 1)

    F_filtered = F * mask
    recon = np.real(ifft2c(F_filtered))

    recon_u8 = norm_to_u8(recon)
    mask_u8 = norm_to_u8(mask)

    return recon_u8, mask_u8, np.log1p(np.abs(F)), py, px

In [9]:
# функции для пересвеченных снимков

def is_overexposed_by_histogram(img_u8, white_level=245, max_white_fraction=0.002):
    """
    Проверяем пересвет по гистограмме
    """
    img = img_u8.astype(np.uint8)
    white_fraction = np.mean(img >= white_level)
    return white_fraction > max_white_fraction


def equalize_overexposed_histogram(img_u8):
    """
    Выравниваем яркость пересвеченного снимка
    """
    img = img_u8.astype(np.uint8)

    # ограничиваем крайние значения, чтобы пересвет не доминировал
    p_low, p_high = np.percentile(img, [0.5, 99.5])
    img_clip = np.clip(img, p_low, p_high)

    # растяжение яркости в диапазон 0..255
    img_norm = cv2.normalize(
        img_clip,
        None,
        alpha=0,
        beta=255,
        norm_type=cv2.NORM_MINMAX
    ).astype(np.uint8)

    # локальное выравнивание гистограммы
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(12, 12))
    img_eq = clahe.apply(img_norm)

    return img_eq

In [16]:
# функции для отрисовки графиков

def robust_common_limits(a: np.ndarray, b: np.ndarray, q_low=1, q_high=99):
    data = np.concatenate([a[np.isfinite(a)].ravel(), b[np.isfinite(b)].ravel()])
    return np.percentile(data, [q_low, q_high])

def wrap_to_pi(x):
    return (x + np.pi) % (2 * np.pi) - np.pi

def align_phase_to_reference(phase_ref, phase_target):
    """
    Приводит phase_target к той же ветви развёртки что и phase_ref
    """
    delta = wrap_to_pi(phase_target - phase_ref)
    return phase_ref + delta

def plot_phase_before_after(phase_before, phase_after, save_path: Path, mode):
    phase_after_aligned = align_phase_to_reference(phase_before, phase_after)

    vmin, vmax = robust_common_limits(phase_before, phase_after_aligned, 1, 99)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    im0 = axes[0].imshow(phase_before, cmap="jet", vmin=vmin, vmax=vmax)
    if mode == 'spherical':
        axes[0].set_title("До улучшения\nфаза после сферического детренда")
    else:
        axes[0].set_title("До улучшения\nфаза после линейного детренда")
    axes[0].set_xlabel("X, px")
    axes[0].set_ylabel("Y, px")

    im1 = axes[1].imshow(phase_after_aligned, cmap="jet", vmin=vmin, vmax=vmax)
    if mode == 'spherical':
        axes[1].set_title("После улучшения\nфаза после сферического детренда")
    else:
        axes[1].set_title("После улучшения\nфаза после линейного детренда")
    axes[1].set_xlabel("X, px")
    axes[1].set_ylabel("Y, px")

    cbar = fig.colorbar(im1, ax=axes, fraction=0.046, pad=0.04)
    cbar.set_label("Фаза, рад")

    fig.savefig(save_path, dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)

def plot_phase_difference(phase_diff_wrapped, save_path: Path, mode):
    fig, ax = plt.subplots(figsize=(7, 6))
    diff_show = phase_diff_wrapped.copy()

    lim = np.nanpercentile(np.abs(diff_show), 99)
    lim = max(lim, 1e-6)

    im = ax.imshow(diff_show, cmap="seismic", vmin=-lim, vmax=lim)
    ax.set_title("Разность фаз: после - до\n")
    ax.set_xlabel("X, px")
    ax.set_ylabel("Y, px")

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Δ фазы, рад")

    fig.savefig(OUTPUT_DIR / "6_phase_difference.png", dpi=PLOT_DPI, bbox_inches="tight")
    plt.close(fig)

def plot_original_mask_result(original_img, mask, result_img, save_path=None, figsize=(18, 5)):
    fig, axes = plt.subplots(1, 3, figsize=figsize)

    axes[0].imshow(original_img, cmap='gray')
    axes[0].set_title('Исходная интерферограмма')
    axes[0].axis('off')

    axes[1].imshow(mask, cmap='gray')
    axes[1].set_title('Маска дефектов')
    axes[1].axis('off')

    axes[2].imshow(result_img, cmap='gray')
    axes[2].set_title('Результат улучшения')
    axes[2].axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

def plot_original_result(original_img, result_img, save_path=None, figsize=(12, 5)):
    fig, axes = plt.subplots(1, 2, figsize=figsize)

    axes[0].imshow(original_img, cmap='gray')
    axes[0].set_title('Локальный дефект до улучшения')
    axes[0].axis('off')

    axes[1].imshow(result_img, cmap='gray')
    axes[1].set_title('Локальный дефект после улучшения')
    axes[1].axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [11]:
# расчет метрик

def fringe_visibility(img):
    img = img.astype(np.float32)
    i_max = np.percentile(img, 99)
    i_min = np.percentile(img, 1)
    return (i_max - i_min) / (i_max + i_min + 1e-8)

def snr_db(img):
    img = img.astype(np.float32)
    mu = np.mean(img)
    sigma = np.std(img)
    return 20 * np.log10(mu / (sigma + 1e-8))

def psd_mean(img, center_exclusion=20):
    img = img.astype(np.float32)
    F = np.fft.fftshift(np.fft.fft2(img))
    psd = np.abs(F) ** 2

    h, w = img.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.ogrid[:h, :w]
    mask = (yy - cy) ** 2 + (xx - cx) ** 2 > center_exclusion ** 2

    return np.mean(psd[mask])

def wrap_phase_diff(d):
    return np.angle(np.exp(1j * d))

def count_phase_residues(phase_wrapped):
    p = phase_wrapped.astype(np.float32)

    d1 = wrap_phase_diff(p[:-1, 1:] - p[:-1, :-1])
    d2 = wrap_phase_diff(p[1:, 1:] - p[:-1, 1:])
    d3 = wrap_phase_diff(p[1:, :-1] - p[1:, 1:])
    d4 = wrap_phase_diff(p[:-1, :-1] - p[1:, :-1])

    residue_sum = d1 + d2 + d3 + d4

    positive = np.sum(residue_sum > np.pi)
    negative = np.sum(residue_sum < -np.pi)

    return int(positive + negative)

def calculate_metrics(img, phase_wrapped=None):
    metrics = {
        "Visibility": fringe_visibility(img),
        "SNR_dB": snr_db(img),
        "PSD_mean": psd_mean(img),
    }

    if phase_wrapped is not None:
        metrics["Phase_residues"] = count_phase_residues(phase_wrapped)

    return metrics

In [24]:
# сам алгоритм

# закрашиваем темные пятна
img_u8 = cv2.imread(INPUT_PATH, cv2.IMREAD_GRAYSCALE)
if img_u8 is None:
    raise FileNotFoundError(f"Не найден файл: {INPUT_PATH}")

mask_before = detect_dark_spots_mask(img_u8)

if OVEREXPOSED:
    img_for_detection = equalize_overexposed_histogram(img_u8)
else:
    img_for_detection = img_u8.copy()

mask = detect_dark_spots_mask(img_for_detection)

result_u8, mag = fill_dark_spots_by_stripe_reconstruction(img_u8, mask)

overlay = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2BGR)
overlay[mask > 0] = (0, 0, 255)

cv2.imwrite(str(OUTPUT_DIR / "0_original.png"), img_u8)
if OVEREXPOSED:
    cv2.imwrite(str(OUTPUT_DIR / "0_1_preprocessed.png"), img_for_detection)
    cv2.imwrite(str(OUTPUT_DIR / "0_2_mask_before_preprocess.png"), (mask_before * 255).astype(np.uint8))
cv2.imwrite(str(OUTPUT_DIR / "1_mask.png"), (mask * 255).astype(np.uint8))
cv2.imwrite(str(OUTPUT_DIR / "2_overlay.png"), overlay)
cv2.imwrite(str(OUTPUT_DIR / "3_fft_log.png"), norm_to_u8(mag))
cv2.imwrite(str(OUTPUT_DIR / "4_result.png"), result_u8)

plot_original_mask_result(original_img=img_u8,
                          mask=(mask_before * 255).astype(np.uint8),
                          result_img=result_u8,
                          save_path=OUTPUT_DIR / "original_mask_result.png")

# расчет фазы после детренда до / после закрашивания
recon_before = reconstruct_complex_field_double_fft(
    img_u8.astype(np.float32),
    lobe_radius=PHASE_LOBE_RADIUS,
    center_exclusion_radius=CENTER_EXCLUSION_RADIUS,
    use_spherical_detrend=(DETREND_MODE == "spherical"),
    fit_tilt=FIT_TILT_AND_OFFSET,
)
recon_after = reconstruct_complex_field_double_fft(
    result_u8.astype(np.float32),
    lobe_radius=PHASE_LOBE_RADIUS,
    center_exclusion_radius=CENTER_EXCLUSION_RADIUS,
    use_spherical_detrend=(DETREND_MODE == "spherical"),
    fit_tilt=FIT_TILT_AND_OFFSET,
)

# комплексные поля после БПФ
field_before = recon_before["complex_field"]
field_after = recon_after["complex_field"]

phase_diff_wrapped, phase_diff_unwrapped = safe_phase_difference_from_fields(field_before, field_after)

phase_before_raw = recon_before["unwrapped_phase"]
phase_after_raw = recon_after["unwrapped_phase"]

if DETREND_MODE == "none":
    phase_before = phase_before_raw
    phase_after = phase_after_raw

elif DETREND_MODE == "linear":
    trend, _ = fit_linear_trend(phase_before_raw)
    phase_before = phase_before_raw - trend
    phase_after = phase_after_raw - trend

elif DETREND_MODE == "spherical":
    trend, _, _ = fit_quadratic_background_robust(
        phase_before_raw,
        fit_tilt=FIT_TILT_AND_OFFSET,
    )
    phase_before = phase_before_raw - trend
    phase_after = phase_after_raw - trend

# расчет метрик качества до и после улучшения изображения
metrics_before = calculate_metrics(img_u8, phase_wrapped=np.angle(field_before))
metrics_after = calculate_metrics(result_u8, phase_wrapped=np.angle(field_after))

metrics_df = pd.DataFrame({
    "Метрика": metrics_before.keys(),
    "До обработки": metrics_before.values(),
    "После обработки": metrics_after.values(),
})

metrics_df.to_csv(OUTPUT_DIR / "metrics_before_after.csv", index=False, encoding="utf-8-sig")

print("\nМетрики качества интерферограмм:")
print("До обработки:")
for k, v in metrics_before.items():
    print(f"  {k}: {v:.4f}")

print("После обработки:")
for k, v in metrics_after.items():
    print(f"  {k}: {v:.4f}")

plot_phase_before_after(phase_before, phase_after, OUTPUT_DIR / "5_phase_before_after.png", mode=DETREND_MODE)
plot_phase_difference(phase_diff_wrapped, OUTPUT_DIR / "6_phase_difference.png", mode=DETREND_MODE)

print(f"Алгоритм отработал. Результаты в папке: {OUTPUT_DIR}")


Метрики качества интерферограмм:
До обработки:
  Visibility: 0.4471
  SNR_dB: 12.8797
  PSD_mean: 139418576.0000
  Phase_residues: 1195.0000
После обработки:
  Visibility: 0.4471
  SNR_dB: 12.9183
  PSD_mean: 138011168.0000
  Phase_residues: 1136.0000
Алгоритм отработал. Результаты в папке: /content/result_stripes_44б


In [20]:
before = cv2.imread('/content/before.png', cv2.IMREAD_GRAYSCALE)
after = cv2.imread('/content/after.png', cv2.IMREAD_GRAYSCALE)

plot_original_result(original_img=before,
                    result_img=after,
                    save_path=OUTPUT_DIR / "original_result.png")

In [21]:
INPUT_PATH = "/content/Img1_1.bmp"
OUTPUT_DIR = Path("/content/result_stripes_Img1_1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BG_SIGMA = 35.0
DARK_SIGMA = 4.0
THR_K = 0.8

MIN_AREA = 20
MAX_AREA = 550
MIN_AREA_AFTER_DILATE = 40
DETREND_MODE = "spherical"
OVEREXPOSED = False

dc_radius = 35
lobe_radius = 32
keep_dc = True

In [23]:
INPUT_PATH = "/content/44б.bmp"
OUTPUT_DIR = Path("/content/result_stripes_44б")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BG_SIGMA = 25.0
DARK_SIGMA = 3.0
THR_K = 1.1
MIN_AREA = 15
MAX_AREA = 580
MASK_DILATE = 1
MIN_AREA_AFTER_DILATE = 50
DETREND_MODE = "linear"
OVEREXPOSED = False

In [ ]:
INPUT_PATH = "/content/1.bmp"
OUTPUT_DIR = Path("/content/result_stripes_1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BG_SIGMA = 9.0
DARK_SIGMA = 1.2
THR_K = 1.6
MIN_AREA = 15
MAX_AREA = 250
MASK_DILATE = 1
MIN_AREA_AFTER_DILATE = 50
DETREND_MODE = "linear"
OVEREXPOSED = False

In [ ]:
INPUT_PATH = "/content/video_1.bmp"
OUTPUT_DIR = Path("/content/result_stripes_video_1")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Параметры поиска темных пятен
BG_SIGMA = 13.0
DARK_SIGMA = 1.0
THR_K = 2.3
MIN_AREA = 50
MAX_AREA = 2500
MASK_DILATE = 1
MIN_AREA_AFTER_DILATE = 20

DETREND_MODE = "linear"
OVEREXPOSED = False

In [ ]:
INPUT_PATH = "/content/50b.bmp"
OUTPUT_DIR = Path("/content/result_stripes_50b")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BG_SIGMA = 45.0
DARK_SIGMA = 2.2
THR_K = 0.85
MIN_AREA = 20
MAX_AREA = 800
MASK_DILATE = 2
MIN_AREA_AFTER_DILATE = 60

DETREND_MODE = "spherical"
OVEREXPOSED = True